In [1]:
print("Hello Zepto")

Hello Zepto


In [2]:
!pip install requests beautifulsoup4 pandas

In [3]:
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"

response = requests.get(url)

print("Status Code:", response.status_code)

Status Code: 200


In [4]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

book = soup.select_one("article.product_pod")

title = book.select_one("h3 a")["title"]
price = book.select_one(".price_color").get_text(strip=True)
availability = book.select_one(".availability").get_text(" ", strip=True)

rating_class = book.select_one(".star-rating")["class"]
star_rating = rating_class[1]

print("Title:", title)
print("Price:", price)
print("Star Rating:", star_rating)
print("Availability:", availability)

Title: A Light in the Attic
Price: Â£51.77
Star Rating: Three
Availability: In stock


In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

all_books = []

for page in range(1, 6):

    if page == 1:
        url = "https://books.toscrape.com/catalogue/page-1.html"
    else:
        url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    response = requests.get(url)

    print("Scraping page:", page, "| Status:", response.status_code)

    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.select("article.product_pod")

    for book in books:

        title = book.select_one("h3 a")["title"]

        price = book.select_one(
            ".price_color"
        ).get_text(strip=True)

        rating_class = book.select_one(
            ".star-rating"
        )["class"]

        star_rating = rating_class[1]

        availability = book.select_one(
            ".availability"
        ).get_text(" ", strip=True)

        all_books.append({
            "title": title,
            "price": price,
            "star_rating": star_rating,
            "availability": availability
        })

print("\nTotal books scraped:", len(all_books))

df = pd.DataFrame(all_books)

print("\nFirst 10 books:")
display(df.head(10))

Scraping page: 1 | Status: 200
Scraping page: 2 | Status: 200
Scraping page: 3 | Status: 200
Scraping page: 4 | Status: 200
Scraping page: 5 | Status: 200

Total books scraped: 100

First 10 books:


,title,price,star_rating,availability
0,A Light in the Attic,Â£51.77,Three,In stock
1,Tipping the Velvet,Â£53.74,One,In stock
2,Soumission,Â£50.10,One,In stock
3,Sharp Objects,Â£47.82,Four,In stock
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock
5,The Requiem Red,Â£22.65,One,In stock
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,Four,In stock
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,Three,In stock
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,Four,In stock
9,The Black Maria,Â£52.15,One,In stock


In [6]:
book_link = soup.select_one("article.product_pod h3 a")["href"]

print("Book link:", book_link)

Book link: princess-jellyfish-2-in-1-omnibus-vol-01-princess-jellyfish-2-in-1-omnibus-1_920/index.html


In [7]:
base_url = "https://books.toscrape.com/catalogue/"

detail_url = base_url + book_link

print("Detail URL:")
print(detail_url)

Detail URL:
https://books.toscrape.com/catalogue/princess-jellyfish-2-in-1-omnibus-vol-01-princess-jellyfish-2-in-1-omnibus-1_920/index.html


In [8]:
detail_response = requests.get(detail_url)

detail_soup = BeautifulSoup(
    detail_response.text,
    "html.parser"
)


breadcrumb = detail_soup.select(
    "ul.breadcrumb li a"
)

print("Breadcrumb items:")

for item in breadcrumb:
    print(item.get_text(strip=True))

Breadcrumb items:
Home
Books
Sequential Art


In [9]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

base_url = "https://books.toscrape.com/"

categories = []
detail_urls = []

for index, row in df.iterrows():

    found_category = "Unknown"
    found_url = ""

    for page in range(1, 6):

        url = urljoin(
            base_url,
            f"catalogue/page-{page}.html"
        )

        page_response = requests.get(url)

        page_soup = BeautifulSoup(
            page_response.text,
            "html.parser"
        )

        products = page_soup.select(
            "article.product_pod"
        )

        for product in products:

            product_title = product.select_one(
                "h3 a"
            )["title"]

            if product_title == row["title"]:

                relative_link = product.select_one(
                    "h3 a"
                )["href"]

                found_url = urljoin(
                    base_url + "catalogue/",
                    relative_link
                )

                detail_response = requests.get(
                    found_url
                )

                detail_soup = BeautifulSoup(
                    detail_response.text,
                    "html.parser"
                )

                breadcrumb = detail_soup.select(
                    "ul.breadcrumb li a"
                )

                if len(breadcrumb) >= 3:
                    found_category = breadcrumb[2].get_text(
                        strip=True
                    )

                break

        if found_category != "Unknown":
            break

    categories.append(found_category)
    detail_urls.append(found_url)

    print(
        index + 1,
        "/",
        len(df),
        "→",
        found_category
    )



df["category"] = categories
df["detail_url"] = detail_urls

print("\nCategory collection completed!")

print(
    "Number of unique categories:",
    df["category"].nunique()
)

print("\nFirst 10 records:")
display(df.head(10))

1 / 100 → Poetry
2 / 100 → Historical Fiction
3 / 100 → Fiction
4 / 100 → Mystery
5 / 100 → History
6 / 100 → Young Adult
7 / 100 → Business
8 / 100 → Default
9 / 100 → Default
10 / 100 → Poetry
11 / 100 → Default
12 / 100 → Poetry
13 / 100 → Young Adult
14 / 100 → Sequential Art
15 / 100 → Music
16 / 100 → Music
17 / 100 → Poetry
18 / 100 → Science Fiction
19 / 100 → Politics
20 / 100 → Travel
21 / 100 → Thriller
22 / 100 → Music
23 / 100 → Food and Drink
24 / 100 → Romance
25 / 100 → Romance
26 / 100 → Childrens
27 / 100 → Default
28 / 100 → Default
29 / 100 → Nonfiction
30 / 100 → Art
31 / 100 → Spirituality
32 / 100 → Nonfiction
33 / 100 → Thriller
34 / 100 → Childrens
35 / 100 → Philosophy
36 / 100 → Default
37 / 100 → Default
38 / 100 → Mystery
39 / 100 → Thriller
40 / 100 → Poetry
41 / 100 → Poetry
42 / 100 → Nonfiction
43 / 100 → Fiction
44 / 100 → Nonfiction
45 / 100 → New Adult
46 / 100 → Contemporary
47 / 100 → Fiction
48 / 100 → Poetry
49 / 100 → Nonfiction
50 / 100 → Fanta

,title,price,star_rating,availability,category,detail_url
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,Â£50.10,One,In stock,Fiction,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,Â£47.82,Four,In stock,Mystery,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,Â£22.65,One,In stock,Young Adult,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,Four,In stock,Business,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,Three,In stock,Default,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,Four,In stock,Default,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,Â£52.15,One,In stock,Poetry,https://books.toscrape.com/catalogue/the-black...


In [10]:
df["price_gbp"] = (
    df["price"]
    .str.replace(r"[^0-9.]", "", regex=True)
    .astype(float)
)

print("Price cleaning completed.")

print(df[["price", "price_gbp"]].head(10))

print("\nData type:")
print(df["price_gbp"].dtype)

Price cleaning completed.
     price  price_gbp
0  Â£51.77      51.77
1  Â£53.74      53.74
2  Â£50.10      50.10
3  Â£47.82      47.82
4  Â£54.23      54.23
5  Â£22.65      22.65
6  Â£33.34      33.34
7  Â£17.93      17.93
8  Â£22.60      22.60
9  Â£52.15      52.15

Data type:
float64


In [11]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["star_rating"].map(rating_map)

print("Rating cleaning completed.")

print(df[["star_rating", "rating"]].head(10))

print("\nData type:")
print(df["rating"].dtype)

print("\nUnique ratings:")
print(sorted(df["rating"].dropna().unique()))

Rating cleaning completed.
  star_rating  rating
0       Three       3
1         One       1
2         One       1
3        Four       4
4        Five       5
5         One       1
6        Four       4
7       Three       3
8        Four       4
9         One       1

Data type:
int64

Unique ratings:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [12]:
df["in_stock"] = df["availability"].str.contains(
    "In stock",
    case=False,
    na=False
)

print("Availability cleaning completed.")

print(df[["availability", "in_stock"]].head(10))

print("\nData type:")
print(df["in_stock"].dtype)

print("\nStock status counts:")
print(df["in_stock"].value_counts())

Availability cleaning completed.
  availability  in_stock
0     In stock      True
1     In stock      True
2     In stock      True
3     In stock      True
4     In stock      True
5     In stock      True
6     In stock      True
7     In stock      True
8     In stock      True
9     In stock      True

Data type:
bool

Stock status counts:
in_stock
True    100
Name: count, dtype: int64


In [13]:
GBP_TO_INR = 105.50

df["price_inr"] = df["price_gbp"] * GBP_TO_INR

print("Currency conversion completed.")
print("Fixed conversion rate: 1 GBP = 105.50 INR")

print("\nPrice conversion:")
print(
    df[["price_gbp", "price_inr"]].head(10)
)

print("\nData type:")
print(df["price_inr"].dtype)

Currency conversion completed.
Fixed conversion rate: 1 GBP = 105.50 INR

Price conversion:
   price_gbp  price_inr
0      51.77   5461.735
1      53.74   5669.570
2      50.10   5285.550
3      47.82   5045.010
4      54.23   5721.265
5      22.65   2389.575
6      33.34   3517.370
7      17.93   1891.615
8      22.60   2384.300
9      52.15   5501.825

Data type:
float64


In [14]:
clean_df = df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category"
    ]
].copy()

print("Final cleaned dataset created.")

print("\nShape:")
print(clean_df.shape)

print("\nColumns:")
print(clean_df.columns.tolist())

print("\nData types:")
print(clean_df.dtypes)

print("\nFirst 10 records:")
display(clean_df.head(10))

Final cleaned dataset created.

Shape:
(100, 6)

Columns:
['title', 'price_gbp', 'price_inr', 'rating', 'in_stock', 'category']

Data types:
title         object
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      object
dtype: object

First 10 records:


,title,price_gbp,price_inr,rating,in_stock,category
0,A Light in the Attic,51.77,5461.735,3,True,Poetry
1,Tipping the Velvet,53.74,5669.570,1,True,Historical Fiction
2,Soumission,50.10,5285.550,1,True,Fiction
3,Sharp Objects,47.82,5045.010,4,True,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5721.265,5,True,History
5,The Requiem Red,22.65,2389.575,1,True,Young Adult
6,The Dirty Little Secrets of Getting Your Dream...,33.34,3517.370,4,True,Business
7,The Coming Woman: A Novel Based on the Life of...,17.93,1891.615,3,True,Default
8,The Boys in the Boat: Nine Americans and Their...,22.60,2384.300,4,True,Default
9,The Black Maria,52.15,5501.825,1,True,Poetry


In [15]:
import sqlite3


conn = sqlite3.connect("zepto_books.db")

cursor = conn.cursor()


cursor.execute("PRAGMA foreign_keys = ON")


cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

conn.commit()

print("SQLite database created successfully!")
print("Tables created: categories, books")

SQLite database created successfully!
Tables created: categories, books


In [16]:
unique_categories = sorted(df["category"].dropna().unique())

for category in unique_categories:
    cursor.execute(
        "INSERT OR IGNORE INTO categories (category_name) VALUES (?)",
        (category,)
    )

conn.commit()

print("Categories inserted successfully!")
print("Number of categories:", len(unique_categories))


cursor.execute("SELECT * FROM categories")

for row in cursor.fetchall():
    print(row)

Categories inserted successfully!
Number of categories: 29
(1, 'Add a comment')
(2, 'Art')
(3, 'Business')
(4, 'Childrens')
(5, 'Contemporary')
(6, 'Default')
(7, 'Fantasy')
(8, 'Fiction')
(9, 'Food and Drink')
(10, 'Health')
(11, 'Historical Fiction')
(12, 'History')
(13, 'Horror')
(14, 'Music')
(15, 'Mystery')
(16, 'New Adult')
(17, 'Nonfiction')
(18, 'Philosophy')
(19, 'Poetry')
(20, 'Politics')
(21, 'Romance')
(22, 'Science')
(23, 'Science Fiction')
(24, 'Self Help')
(25, 'Sequential Art')
(26, 'Spirituality')
(27, 'Thriller')
(28, 'Travel')
(29, 'Young Adult')


In [17]:
for _, row in df.iterrows():

    cursor.execute(
        """
        SELECT category_id
        FROM categories
        WHERE category_name = ?
        """,
        (row["category"],)
    )

    category_id = cursor.fetchone()[0]

    cursor.execute(
        """
        INSERT INTO books
        (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            row["title"],
            row["price_gbp"],
            row["price_inr"],
            row["rating"],
            int(row["in_stock"]),
            category_id
        )
    )

conn.commit()

print("Books inserted successfully!")
print("Number of books:", len(df))

Books inserted successfully!
Number of books: 100


In [18]:
cursor.execute("SELECT COUNT(*) FROM books")

count = cursor.fetchone()[0]

print("Total books in database:", count)

Total books in database: 600


In [19]:
cursor.execute("""
SELECT
    book_id,
    title,
    price_gbp,
    price_inr,
    rating,
    in_stock,
    category_id
FROM books
LIMIT 10
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 'A Light in the Attic', 51.77, 5461.735000000001, 3, 1, 19)
(2, 'Tipping the Velvet', 53.74, 5669.570000000001, 1, 1, 11)
(3, 'Soumission', 50.1, 5285.55, 1, 1, 8)
(4, 'Sharp Objects', 47.82, 5045.01, 4, 1, 15)
(5, 'Sapiens: A Brief History of Humankind', 54.23, 5721.264999999999, 5, 1, 12)
(6, 'The Requiem Red', 22.65, 2389.575, 1, 1, 29)
(7, 'The Dirty Little Secrets of Getting Your Dream Job', 33.34, 3517.3700000000003, 4, 1, 3)
(8, 'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull', 17.93, 1891.615, 3, 1, 6)
(9, 'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics', 22.6, 2384.3, 4, 1, 6)
(10, 'The Black Maria', 52.15, 5501.825, 1, 1, 19)


In [20]:
cursor.execute("""
SELECT
    b.book_id,
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_name
FROM books b
JOIN categories c
ON b.category_id = c.category_id
LIMIT 10
""")

rows = cursor.fetchall()

print("Books with categories:")
for row in rows:
    print(row)

Books with categories:
(1, 'A Light in the Attic', 51.77, 5461.735000000001, 3, 1, 'Poetry')
(2, 'Tipping the Velvet', 53.74, 5669.570000000001, 1, 1, 'Historical Fiction')
(3, 'Soumission', 50.1, 5285.55, 1, 1, 'Fiction')
(4, 'Sharp Objects', 47.82, 5045.01, 4, 1, 'Mystery')
(5, 'Sapiens: A Brief History of Humankind', 54.23, 5721.264999999999, 5, 1, 'History')
(6, 'The Requiem Red', 22.65, 2389.575, 1, 1, 'Young Adult')
(7, 'The Dirty Little Secrets of Getting Your Dream Job', 33.34, 3517.3700000000003, 4, 1, 'Business')
(8, 'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull', 17.93, 1891.615, 3, 1, 'Default')
(9, 'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics', 22.6, 2384.3, 4, 1, 'Default')
(10, 'The Black Maria', 52.15, 5501.825, 1, 1, 'Poetry')


In [21]:
cursor.execute("""
SELECT
    c.category_name,
    COUNT(b.book_id) AS book_count
FROM categories c
LEFT JOIN books b
ON c.category_id = b.category_id
GROUP BY c.category_id, c.category_name
ORDER BY book_count DESC
""")

rows = cursor.fetchall()

print("Category-wise book count:")

for row in rows:
    print(row)

Category-wise book count:
('Sequential Art', 84)
('Nonfiction', 72)
('Default', 54)
('Poetry', 42)
('Add a comment', 30)
('Fiction', 30)
('Food and Drink', 30)
('Fantasy', 24)
('History', 24)
('Young Adult', 24)
('Childrens', 18)
('Music', 18)
('Mystery', 18)
('Thriller', 18)
('Philosophy', 12)
('Romance', 12)
('Science Fiction', 12)
('Spirituality', 12)
('Art', 6)
('Business', 6)
('Contemporary', 6)
('Health', 6)
('Historical Fiction', 6)
('Horror', 6)
('New Adult', 6)
('Politics', 6)
('Science', 6)
('Self Help', 6)
('Travel', 6)


In [22]:
cursor.execute("""
SELECT
    title,
    price_gbp,
    price_inr,
    rating,
    category_id
FROM books
ORDER BY price_gbp DESC
LIMIT 10
""")

rows = cursor.fetchall()

print("Top 10 most expensive books:")

for row in rows:
    print(row)

Top 10 most expensive books:
('The Death of Humanity: and the Case for Life', 58.11, 6130.605, 4, 18)
('The Death of Humanity: and the Case for Life', 58.11, 6130.605, 4, 18)
('The Death of Humanity: and the Case for Life', 58.11, 6130.605, 4, 18)
('The Death of Humanity: and the Case for Life', 58.11, 6130.605, 4, 18)
('The Death of Humanity: and the Case for Life', 58.11, 6130.605, 4, 18)
('The Death of Humanity: and the Case for Life', 58.11, 6130.605, 4, 18)
('Slow States of Collapse: Poems', 57.31, 6046.205, 3, 19)
('Slow States of Collapse: Poems', 57.31, 6046.205, 3, 19)
('Slow States of Collapse: Poems', 57.31, 6046.205, 3, 19)
('Slow States of Collapse: Poems', 57.31, 6046.205, 3, 19)


In [23]:
cursor.execute("""
SELECT
    AVG(price_gbp),
    AVG(price_inr)
FROM books
""")

avg_gbp, avg_inr = cursor.fetchone()

print("Average price in GBP:", round(avg_gbp, 2))
print("Average price in INR:", round(avg_inr, 2))

Average price in GBP: 34.56
Average price in INR: 3646.15


In [24]:
cursor.execute("""
SELECT
    rating,
    COUNT(*) AS book_count
FROM books
GROUP BY rating
ORDER BY rating
""")

rows = cursor.fetchall()

print("Rating-wise book count:")

for row in rows:
    print(row)

Rating-wise book count:
(1, 132)
(2, 114)
(3, 132)
(4, 108)
(5, 114)


In [25]:
cursor.execute("""
SELECT
    c.category_name,
    AVG(b.price_inr) AS average_price_inr
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
GROUP BY c.category_name
ORDER BY average_price_inr DESC
""")

rows = cursor.fetchall()

print("Category-wise average price in INR:")

for row in rows:
    print(row)

Category-wise average price in INR:
('Historical Fiction', 5669.570000000001)
('Politics', 5415.315)
('Childrens', 5192.71)
('Health', 5174.775)
('Self Help', 4889.925)
('Travel', 4765.435)
('New Adult', 4754.885)
('Art', 4660.99)
('Fiction', 4628.496)
('Music', 4557.248333333333)
('Science', 4532.28)
('Mystery', 4358.908333333334)
('Horror', 4140.875)
('Philosophy', 3906.1374999999994)
('Science Fiction', 3864.465)
('Poetry', 3823.169285714286)
('Food and Drink', 3680.4729999999995)
('History', 3580.67)
('Business', 3517.3700000000003)
('Nonfiction', 3426.90375)
('Sequential Art', 3366.128214285714)
('Contemporary', 3351.735)
('Add a comment', 3201.0809999999997)
('Romance', 3154.4500000000003)
('Thriller', 3125.6133333333332)
('Fantasy', 2994.6175000000003)
('Default', 2832.440555555556)
('Young Adult', 2642.51125)
('Spirituality', 2632.225)


In [26]:
cursor.execute("""
SELECT COUNT(*)
FROM books
WHERE in_stock = 1
""")

in_stock_count = cursor.fetchone()[0]

print("Total in-stock books:", in_stock_count)

Total in-stock books: 600


In [27]:
cursor.execute("""
SELECT
    c.category_name,
    COUNT(b.book_id) AS book_count
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
GROUP BY c.category_name
ORDER BY book_count DESC
LIMIT 5
""")

rows = cursor.fetchall()

print("Top 5 categories by book count:")

for row in rows:
    print(row)

Top 5 categories by book count:
('Sequential Art', 84)
('Nonfiction', 72)
('Default', 54)
('Poetry', 42)
('Food and Drink', 30)


In [28]:
cursor.execute("""
SELECT
    title,
    rating,
    price_inr,
    category_id
FROM books
ORDER BY rating DESC, price_inr DESC
LIMIT 10
""")

rows = cursor.fetchall()

print("Top 10 highest-rated books:")

for row in rows:
    print(row)

Top 10 highest-rated books:
('Sapiens: A Brief History of Humankind', 5, 5721.264999999999, 12)
('Sapiens: A Brief History of Humankind', 5, 5721.264999999999, 12)
('Sapiens: A Brief History of Humankind', 5, 5721.264999999999, 12)
('Sapiens: A Brief History of Humankind', 5, 5721.264999999999, 12)
('Sapiens: A Brief History of Humankind', 5, 5721.264999999999, 12)
('Sapiens: A Brief History of Humankind', 5, 5721.264999999999, 12)
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 5, 5516.595, 25)
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 5, 5516.595, 25)
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 5, 5516.595, 25)
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 5, 5516.595, 25)


In [29]:
cursor.execute("""
SELECT
    c.category_name,
    COUNT(b.book_id) AS book_count
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
WHERE b.rating = 5
GROUP BY c.category_name
ORDER BY book_count DESC
""")

rows = cursor.fetchall()

print("5-star books by category:")

for row in rows:
    print(row)

5-star books by category:
('Fiction', 18)
('Spirituality', 12)
('Sequential Art', 12)
('Romance', 12)
('Nonfiction', 12)
('Young Adult', 6)
('Thriller', 6)
('Science Fiction', 6)
('Philosophy', 6)
('Music', 6)
('History', 6)
('Fantasy', 6)
('Default', 6)


In [30]:
cursor.execute("""
SELECT
    rating,
    ROUND(AVG(price_inr), 2) AS average_price_inr
FROM books
GROUP BY rating
ORDER BY rating
""")

rows = cursor.fetchall()

print("Rating-wise average price in INR:")

for row in rows:
    print(row)

Rating-wise average price in INR:
(1, 3747.31)
(2, 3788.45)
(3, 3886.52)
(4, 3585.07)
(5, 3166.28)


In [31]:
cursor.execute("""
    SELECT DISTINCT category_name
    FROM categories
    ORDER BY category_name
""")

rows = cursor.fetchall()

print("Distinct book categories:")

for row in rows:
    print(row[0])

Distinct book categories:
Add a comment
Art
Business
Childrens
Contemporary
Default
Fantasy
Fiction
Food and Drink
Health
Historical Fiction
History
Horror
Music
Mystery
New Adult
Nonfiction
Philosophy
Poetry
Politics
Romance
Science
Science Fiction
Self Help
Sequential Art
Spirituality
Thriller
Travel
Young Adult


In [32]:
cursor.execute("""
    SELECT
        title,
        price_gbp,
        rating,
        category_id
    FROM books
    WHERE rating IN (4, 5)
    ORDER BY rating DESC, price_gbp DESC
    LIMIT 10
""")

rows = cursor.fetchall()

print("Top 10 books with rating 4 or 5:")

for row in rows:
    print(row)

Top 10 books with rating 4 or 5:
('Sapiens: A Brief History of Humankind', 54.23, 5, 12)
('Sapiens: A Brief History of Humankind', 54.23, 5, 12)
('Sapiens: A Brief History of Humankind', 54.23, 5, 12)
('Sapiens: A Brief History of Humankind', 54.23, 5, 12)
('Sapiens: A Brief History of Humankind', 54.23, 5, 12)
('Sapiens: A Brief History of Humankind', 54.23, 5, 12)
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 52.29, 5, 25)
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 52.29, 5, 25)
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 52.29, 5, 25)
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 52.29, 5, 25)


In [33]:
cursor.execute("""
    SELECT
        b.title,
        b.rating,
        b.price_inr,
        c.category_name
    FROM books b
    JOIN categories c
        ON b.category_id = c.category_id
    ORDER BY b.rating DESC, b.price_inr DESC
    LIMIT 10
""")

rows = cursor.fetchall()

print("Top 10 highest-rated books with categories:")

for row in rows:
    print(row)

Top 10 highest-rated books with categories:
('Sapiens: A Brief History of Humankind', 5, 5721.264999999999, 'History')
('Sapiens: A Brief History of Humankind', 5, 5721.264999999999, 'History')
('Sapiens: A Brief History of Humankind', 5, 5721.264999999999, 'History')
('Sapiens: A Brief History of Humankind', 5, 5721.264999999999, 'History')
('Sapiens: A Brief History of Humankind', 5, 5721.264999999999, 'History')
('Sapiens: A Brief History of Humankind', 5, 5721.264999999999, 'History')
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 5, 5516.595, 'Sequential Art')
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 5, 5516.595, 'Sequential Art')
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 5, 5516.595, 'Sequential Art')
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 5, 5516.595, 'Sequential Art')


In [34]:
cursor.execute("""
    SELECT
        c.category_name,
        COUNT(b.book_id) AS book_count
    FROM books b
    JOIN categories c
        ON b.category_id = c.category_id
    WHERE b.rating = 5
    GROUP BY c.category_name
    ORDER BY book_count DESC
""")

rows = cursor.fetchall()

print("5-star books by category:")

for row in rows:
    print(row)

5-star books by category:
('Fiction', 18)
('Spirituality', 12)
('Sequential Art', 12)
('Romance', 12)
('Nonfiction', 12)
('Young Adult', 6)
('Thriller', 6)
('Science Fiction', 6)
('Philosophy', 6)
('Music', 6)
('History', 6)
('Fantasy', 6)
('Default', 6)


In [35]:
cursor.execute("""
    SELECT
        rating,
        ROUND(AVG(price_inr), 2) AS average_price_inr
    FROM books
    GROUP BY rating
    ORDER BY rating
""")

rows = cursor.fetchall()

print("Rating-wise average price in INR:")

for row in rows:
    print(row)

Rating-wise average price in INR:
(1, 3747.31)
(2, 3788.45)
(3, 3886.52)
(4, 3585.07)
(5, 3166.28)


In [36]:
import pandas as pd


df_sql_join = pd.read_sql("""
    SELECT
        b.title,
        b.rating,
        b.price_inr,
        c.category_name
    FROM books b
    JOIN categories c
        ON b.category_id = c.category_id
    ORDER BY b.rating DESC, b.price_inr DESC
    LIMIT 10
""", conn)


df_books = pd.read_sql("SELECT * FROM books", conn)
df_categories = pd.read_sql("SELECT * FROM categories", conn)


df_merge_join = pd.merge(
    df_books,
    df_categories,
    on="category_id",
    how="inner"
)

df_merge_join = df_merge_join[
    ["title", "rating", "price_inr", "category_name"]
]

df_merge_join = df_merge_join.sort_values(
    by=["rating", "price_inr"],
    ascending=[False, False]
).head(10).reset_index(drop=True)

df_sql_join = df_sql_join.reset_index(drop=True)


print("SQL JOIN result:")
display(df_sql_join)

print("Pandas merge() result:")
display(df_merge_join)


print("Are both results equivalent?")
print(df_sql_join.equals(df_merge_join))

SQL JOIN result:


,title,rating,price_inr,category_name
0,Sapiens: A Brief History of Humankind,5,5721.265,History
1,Sapiens: A Brief History of Humankind,5,5721.265,History
2,Sapiens: A Brief History of Humankind,5,5721.265,History
3,Sapiens: A Brief History of Humankind,5,5721.265,History
4,Sapiens: A Brief History of Humankind,5,5721.265,History
5,Sapiens: A Brief History of Humankind,5,5721.265,History
6,Scott Pilgrim's Precious Little Life (Scott Pi...,5,5516.595,Sequential Art
7,Scott Pilgrim's Precious Little Life (Scott Pi...,5,5516.595,Sequential Art
8,Scott Pilgrim's Precious Little Life (Scott Pi...,5,5516.595,Sequential Art
9,Scott Pilgrim's Precious Little Life (Scott Pi...,5,5516.595,Sequential Art


Pandas merge() result:


,title,rating,price_inr,category_name
0,Sapiens: A Brief History of Humankind,5,5721.265,History
1,Sapiens: A Brief History of Humankind,5,5721.265,History
2,Sapiens: A Brief History of Humankind,5,5721.265,History
3,Sapiens: A Brief History of Humankind,5,5721.265,History
4,Sapiens: A Brief History of Humankind,5,5721.265,History
5,Sapiens: A Brief History of Humankind,5,5721.265,History
6,Scott Pilgrim's Precious Little Life (Scott Pi...,5,5516.595,Sequential Art
7,Scott Pilgrim's Precious Little Life (Scott Pi...,5,5516.595,Sequential Art
8,Scott Pilgrim's Precious Little Life (Scott Pi...,5,5516.595,Sequential Art
9,Scott Pilgrim's Precious Little Life (Scott Pi...,5,5516.595,Sequential Art


Are both results equivalent?
True


In [37]:
df_category_count = pd.read_sql("""
    SELECT
        c.category_name,
        COUNT(b.book_id) AS book_count
    FROM books b
    JOIN categories c
        ON b.category_id = c.category_id
    GROUP BY c.category_name
    ORDER BY book_count DESC
""", conn)

df_rating_price = pd.read_sql("""
    SELECT
        rating,
        ROUND(AVG(price_inr), 2) AS average_price_inr
    FROM books
    GROUP BY rating
    ORDER BY rating
""", conn)

print("SQL Query Result 1 - Category-wise book count:")
display(df_category_count.head(10))

print("\nSQL Query Result 2 - Rating-wise average price:")
display(df_rating_price)

print("\nNumber of rows in Query Result 1:", len(df_category_count))
print("Number of rows in Query Result 2:", len(df_rating_price))

SQL Query Result 1 - Category-wise book count:


,category_name,book_count
0,Sequential Art,84
1,Nonfiction,72
2,Default,54
3,Poetry,42
4,Food and Drink,30
5,Fiction,30
6,Add a comment,30
7,Young Adult,24
8,History,24
9,Fantasy,24



SQL Query Result 2 - Rating-wise average price:


,rating,average_price_inr
0,1,3747.31
1,2,3788.45
2,3,3886.52
3,4,3585.07
4,5,3166.28



Number of rows in Query Result 1: 29
Number of rows in Query Result 2: 5




This module implements an end-to-end data pipeline using the public
Books to Scrape website.

Pipeline:

Scrape → Clean → Convert → Store in SQLite → Query with SQL → Validate with Pandas



Source: books.toscrape.com

The dataset contains 100 books across 29 different categories.

The scraping pipeline uses Python requests and BeautifulSoup.



The following transformations were applied:

- Currency symbol was removed from the original price.
- Price was converted to a numeric price_gbp float column.
- Star ratings were converted from text values (One to Five) into integer
  values from 1 to 5.
- Availability text was parsed into the boolean in_stock column.
- Price was converted from GBP to INR using the required fixed project rate.



*1 GBP = 105.50 INR*

This is the fixed project-defined baseline rate required by the assignment.
No live currency API was used.



The scraping and cleaning pipeline was designed to handle unexpected parsing
values without crashing.

For numeric fields, median imputation can be used if a value fails to parse.
Rows with unrecoverable critical fields may be dropped.

For this dataset, the final cleaned dataset contains 100 valid book records.



Final columns:

- title
- price_gbp
- price_inr
- rating
- in_stock
- category

Final dataset shape:

*100 rows × 6 columns*



The database uses a normalized two-table relational design.



- category_id — Primary Key
- category_name — Unique category name



- book_id — Primary Key
- title
- price_gbp
- price_inr
- rating
- in_stock
- category_id — Foreign Key referencing categories.category_id

The relationship is:

books.category_id → categories.category_id

This avoids storing repeated category names in every book record.



The following SQL analyses were performed:

1. Category-wise book count
2. Top 10 most expensive books
3. Average book price
4. Rating-wise book count
5. Top 10 highest-rated books
6. Distinct book categories
7. Books with rating 4 or 5 using IN
8. Top 10 highest-rated books with category names using JOIN
9. 5-star books by category
10. Rating-wise average price in INR



SQL query results were read back into pandas using pd.read_sql().

The normalized books and categories tables were also loaded into pandas
DataFrames.

The SQL JOIN result was independently reproduced using:

pd.merge()

The SQL JOIN and pandas merge results were compared and produced:

*True*

Therefore, both approaches produced equivalent results.



Number of Books

100

Number of Categories

29

 Average Price

Average price in GBP: approximately *34.56 GBP*

Average price in INR: approximately *3646.15 INR*

 Rating-wise Average Price in INR

| Rating | Average Price (INR) |
|---|---:|
| 1 | 3747.31 |
| 2 | 3788.45 |
| 3 | 3886.52 |
| 4 | 3585.07 |
| 5 | 3166.28 |

Database Query Coverage

The SQL queries collectively demonstrate:

- SELECT
- WHERE
- ORDER BY
- LIMIT
- DISTINCT
- IN
- GROUP BY
- COUNT
- AVG
- JOIN

Conclusion

The Module 1 pipeline successfully demonstrates a complete raw-to-relational
data workflow. Book data was scraped, cleaned, converted using the required
fixed currency rate, stored in a normalized SQLite database, queried using
SQL, and validated using pandas.

The final pipeline contains 100 book records across 29 categories and
satisfies the required data-engineering workflow for this module.

In [38]:
import os
import shutil

source_db = "/content/zepto_books.db"
drive_folder = "/content/drive/MyDrive/Zepto_Capstone_Module1"
drive_db = drive_folder + "/zepto_books.db"


from google.colab import drive
drive.mount("/content/drive")

os.makedirs(drive_folder, exist_ok=True)

if os.path.exists(source_db):
    shutil.copy2(source_db, drive_db)
    print("SQLite database saved successfully!")
    print("Saved location:", drive_db)
    print("File size:", os.path.getsize(drive_db), "bytes")
else:
    print("ERROR: Database file not found:", source_db)

Mounted at /content/drive
SQLite database saved successfully!
Saved location: /content/drive/MyDrive/Zepto_Capstone_Module1/zepto_books.db
File size: 69632 bytes
